In [1]:
### 1. Download e Preparação do Dataset
import polars as pl
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
import asyncio
from datasets import Dataset
import os

In [2]:
from dotenv import load_dotenv
from huggingface_hub import login

# Carregar as variáveis de ambiente do .env
load_dotenv()

# Obter o token
hf_token = os.getenv("HF_TOKEN")

if hf_token:
    login(token=hf_token)
else:
    raise ValueError("O token da Hugging Face (HF_TOKEN) não foi encontrado no arquivo .env")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
def load_fine_tuning_dataset():
    """
    Carrega o dataset de fine-tuning a partir de um arquivo JSON onde cada linha contém um objeto JSON separado.
    """
    data = pl.read_ndjson("trn.json")
    return data.select(["title", "content"]).fill_null("")

def load_train_test_datasets():
    """
    Carrega os datasets de treino e teste a partir de arquivos TXT.
    """
    dataset = {}
    for key, filename in zip(["train", "test"],
                              ["filter_labels_train.txt", "filter_labels_test.txt"]):
        data = pl.read_csv(filename, separator="\n", has_header=False).to_pandas()
        dataset[key] = pl.DataFrame({"question": data[0::2].reset_index(drop=True), "answer": data[1::2].reset_index(drop=True)})
    return dataset

dataset_fine_tuning = load_fine_tuning_dataset()
dataset_train_test = load_train_test_datasets()

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)

In [5]:
async def persist_data_to_chromadb(dataset, batch_size=10000):
    """
    Persiste os dados no ChromaDB de forma assíncrona, utilizando inserção em lote.
    """
    texts = (dataset["title"] + " " + dataset["content"]).to_list()
    tasks = [vectorstore.aadd_documents([{"text": text} for text in texts[i:i + batch_size]])
             for i in range(0, len(texts), batch_size)]
    await asyncio.gather(*tasks)

# Função para garantir a execução da corrotina com parâmetros corretos
def run_persist_data_to_chromadb(dataset, batch_size=10000):
    loop = asyncio.get_event_loop()
    
    if loop.is_running():
        # Se o loop já está rodando, criamos a tarefa e a agendamos
        loop.create_task(persist_data_to_chromadb(dataset, batch_size))
    else:
        # Se o loop não estiver rodando, podemos rodar normalmente
        loop.run_until_complete(persist_data_to_chromadb(dataset, batch_size))

# Exemplo de chamada
run_persist_data_to_chromadb(dataset_fine_tuning, batch_size=10000)


In [6]:
### 3. Execução do Fine-Tuning
def fine_tune_model(dataset):
    """
    Realiza o fine-tuning do modelo Llama 3.
    """
    model_name = "meta-llama/Llama-3.3-70B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    
    def tokenize_function(examples):
        return tokenizer(examples["content"], padding="max_length", truncation=True)
    
    train_dataset = Dataset.from_pandas(dataset.to_pandas())
    tokenized_datasets = train_dataset.map(tokenize_function, batched=True)
    
    training_args = TrainingArguments(output_dir="./results", num_train_epochs=3, per_device_train_batch_size=8, logging_dir="./logs")
    trainer = Trainer(model=model, args=training_args, train_dataset=tokenized_datasets)
    
    trainer.train()
    return model

In [7]:
def retrieve_context(question):
    """
    Recupera contexto relevante do ChromaDB com base na pergunta do usuário.
    """
    docs = vectorstore.similarity_search(question, k=3)
    return " ".join([doc.page_content for doc in docs])


In [8]:
def generate_response(question):
    """
    Gera uma resposta com base na pergunta do usuário e no contexto recuperado.
    """
    context = retrieve_context(question)
    prompt = f"Contexto: {context}\nPergunta: {question}\nResposta:"
    
    model_name = "meta-llama/Llama-3.3-70B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, device_map="cpu")
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cpu")
    outputs = model.generate(**inputs)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

In [ ]:
pergunta = "Qual é o melhor fone de ouvido?"
resposta = generate_response(pergunta)
print(f"Resposta: {resposta}")


Loading checkpoint shards:   0%|          | 0/30 [00:00<?, ?it/s]

: 